# HIV Adherence ML Model

This notebook pulls adherence data from Firebase, trains a RandomForest model, and saves predictions back to Firebase.

In [ ]:
!pip install firebase-admin

In [ ]:
import firebase_admin
from firebase_admin import credentials, db
import pandas as pd
import numpy as np

cred = credentials.Certificate('serviceAccountKey.json')
firebase_admin.initialize_app(cred, {
    'databaseURL': 'https://hiv-adherence-tracker-default-rtdb.firebaseio.com'  # Replace with your URL
})
print('Connected to Firebase!')

In [ ]:
def pull_logs(patient_id):
    ref = db.reference(f'adherence_logs/{patient_id}')
    data = ref.get()
    if not data:
        return pd.DataFrame()
    rows = list(data.values())
    df = pd.DataFrame(rows)
    df['taken'] = df['taken'].astype(bool)
    return df

df = pull_logs('P_001')
print(f'Loaded {len(df)} records from Firebase')
print(df.head())

In [ ]:
def compute_features(df):
    if df.empty: return {}
    records = df['taken'].tolist()

    # Feature 1: Overall adherence rate
    adherence_rate = df['taken'].mean()

    # Feature 2: 7-day adherence rate
    recent = df.tail(21)  # last 7 days x 3 doses
    adherence_7d = recent['taken'].mean()

    # Feature 3: Missed streak (consecutive misses)
    streak = 0
    for v in reversed(records):
        if not v: streak += 1
        else: break

    # Feature 4: Total missed doses
    total_missed = int((~df['taken']).sum())

    return {
        'adherence_rate': round(adherence_rate, 3),
        'adherence_7d': round(adherence_7d, 3),
        'missed_streak': streak,
        'total_missed': total_missed,
        'side_effect_freq': 0
    }

feats = compute_features(df)
print('Features:', feats)

In [ ]:
# For single patient testing, we simulate multiple patients
# For real data, loop over all patient IDs

import random
rows = []
for i in range(100):  # 100 simulated patients
    adh = random.uniform(0.5, 1.0)
    streak = random.randint(0, 7)
    adh7 = max(0, adh + random.uniform(-0.15, 0.15))
    missed = int((1 - adh) * 90)
    risk = 1 if adh < 0.75 or streak >= 3 else 0
    rows.append({
        'adherence_rate': adh, 'adherence_7d': adh7,
        'missed_streak': streak, 'total_missed': missed,
        'side_effect_freq': 0, 'risk_label': risk
    })

train_df = pd.DataFrame(rows)
print(train_df.head())
print('Risk distribution:')
print(train_df['risk_label'].value_counts())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

X = train_df[['adherence_rate','adherence_7d','missed_streak','total_missed','side_effect_freq']]
y = train_df['risk_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

joblib.dump(model, 'adherence_model.pkl')
print('Model saved as adherence_model.pkl')

In [ ]:
# Compute features for the real patient from Firebase
feats = compute_features(df)

input_data = [[
    feats['adherence_rate'],
    feats['adherence_7d'],
    feats['missed_streak'],
    feats['total_missed'],
    feats['side_effect_freq']
]]

risk_prob = model.predict_proba(input_data)[0][1]  # probability of high risk
risk_label = 'High' if risk_prob > 0.6 else 'Moderate' if risk_prob > 0.35 else 'Low'

print(f'Risk Score: {risk_prob:.2%}')
print(f'Risk Level: {risk_label}')

# Save prediction back to Firebase so the app can read it
pred_ref = db.reference('predictions/P_001')
pred_ref.set({
    'risk_score': round(float(risk_prob), 3),
    'risk_label': risk_label,
    'adherence_rate': feats['adherence_rate'],
    'adherence_7d': feats['adherence_7d'],
    'missed_streak': feats['missed_streak'],
    'updated_at': str(pd.Timestamp.now())
})
print('Prediction saved to Firebase!')